# Lasso Regression - Scikit-Learn

## What is Lasso Regression?

Lasso (Least Absolute Shrinkage and Selection Operator) Regression is a linear regression with L1 regularization. It adds a penalty equal to the absolute value of the magnitude of coefficients. This forces some coefficients to become exactly zero, performing automatic feature selection.

---

## Key Benefits

| Benefit | Description |
|---------|-------------|
| L1 Regularization | Adds L1 penalty to model weights |
| Feature Selection | Shrinks some coefficients to exactly zero |
| Sparsity | Produces sparse models with fewer features |
| Interpretability | Simplifies model by removing irrelevant features |

---

## Lasso vs Ridge

| Aspect | Ridge (L2) | Lasso (L1) |
|--------|------------|------------|
| Penalty | Sum of squared coefficients | Sum of absolute coefficients |
| Feature Selection | No (keeps all features) | Yes (can zero out features) |
| Solution Type | Closed-form | Coordinate descent |
| Use Case | Multicollinearity | Feature selection, sparse models |

---

## Lasso Formula

$$\text{Loss} = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 + \lambda \sum_{j=1}^{p} |w_j|$$

---

## Applications

| Domain | Use Case |
|--------|----------|
| Feature Selection | Identifying important features |
| High-Dimensional Data | When features exceed samples |
| Genomics | Gene selection in microarray data |
| Signal Processing | Sparse signal reconstruction |

---

## Advantages

- Automatic Feature Selection: Sets some coefficients to zero
- Sparse Models: Simpler and more interpretable
- Handles High Dimensions: Works when p > n
- Reduces Overfitting: Shrinks coefficients

---

## Limitations

- No Closed-Form Solution: Requires iterative methods
- Correlation Issues: Tends to select one of correlated features
- Hyperparameter Sensitivity: Requires careful alpha tuning
- Instability: Can be unstable with highly correlated features

---

## One-Line Summary

**Lasso Regression adds L1 penalty to linear regression, performing feature selection by shrinking some coefficients to zero.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso, Ridge, LassoCV
from sklearn.model_selection import GridSearchCV, train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

print("="*50)
print("LASSO REGRESSION - SCIKIT-LEARN")
print("="*50)

In [ ]:
# Create synthetic dataset
np.random.seed(0)
n_samples = 200
n_features = 10

X = np.random.randn(n_samples, n_features)

# Only first 5 features are important (sparse coefficients)
true_coef = np.array([3.2, -1.5, 0.7, 0, 2.8, 0, 0, 0, 0, 0])
y = X.dot(true_coef) + np.random.randn(n_samples) * 0.6

print("Dataset created:")
print(f"Samples: {n_samples}")
print(f"Features: {n_features}")
print(f"True coefficients (only first 5 non-zero): {true_coef}")

In [ ]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, random_state=42
)

print(f"Train size: {len(X_train)}")
print(f"Test size: {len(X_test)}")

## Linear Regression (Baseline)

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

lr_mse = mean_squared_error(y_test, y_pred_lr)
lr_r2 = r2_score(y_test, y_pred_lr)

print("Linear Regression Results:")
print(f"MSE: {lr_mse:.4f}")
print(f"R2 Score: {lr_r2:.4f}")

## Lasso Regression with Alpha=0.1

In [ ]:
lasso = Lasso(alpha=0.1, max_iter=10000)
lasso.fit(X_train, y_train)
y_pred_lasso = lasso.predict(X_test)

lasso_mse = mean_squared_error(y_test, y_pred_lasso)
lasso_r2 = r2_score(y_test, y_pred_lasso)

print("Lasso Regression Results (alpha=0.1):")
print(f"MSE: {lasso_mse:.4f}")
print(f"R2 Score: {lasso_r2:.4f}")
print(f"Non-zero coefficients: {np.sum(lasso.coef_ != 0)}")
print(f"Coefficients: {lasso.coef_}")

## Lasso vs Ridge Comparison

In [ ]:
ridge = Ridge(alpha=0.1)
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)
ridge_mse = mean_squared_error(y_test, y_pred_ridge)
ridge_r2 = r2_score(y_test, y_pred_ridge)

print("\n" + "="*50)
print("Lasso vs Ridge Comparison")
print("="*50)
print(f"{'Model':<15} {'MSE':<12} {'R2 Score':<12} {'Non-zero':<12}")
print("-"*50)
print(f"{'Linear':<15} {lr_mse:<12.4f} {lr_r2:<12.4f} {n_features:<12}")
print(f"{'Ridge':<15} {ridge_mse:<12.4f} {ridge_r2:<12.4f} {n_features:<12}")
print(f"{'Lasso':<15} {lasso_mse:<12.4f} {lasso_r2:<12.4f} {np.sum(lasso.coef_ != 0):<12}")

## GridSearchCV for Best Alpha

In [ ]:
param_grid = {"alpha": [0.001, 0.01, 0.05, 0.1, 0.5, 1, 5, 10]}
grid = GridSearchCV(Lasso(max_iter=10000), param_grid, cv=5, scoring="neg_mean_squared_error")
grid.fit(X_train, y_train)

best_lasso = grid.best_estimator_
best_alpha = grid.best_params_["alpha"]
y_pred_best = best_lasso.predict(X_test)

best_mse = mean_squared_error(y_test, y_pred_best)
best_r2 = r2_score(y_test, y_pred_best)

print("\nGridSearchCV Results:")
print(f"Best alpha selected: {best_alpha}")
print(f"MSE (best alpha): {best_mse:.4f}")
print(f"R2 Score (best alpha): {best_r2:.4f}")
print(f"Non-zero coefficients: {np.sum(best_lasso.coef_ != 0)}")

## LassoCV (Built-in Cross-Validation)

In [ ]:
lasso_cv = LassoCV(alphas=[0.001, 0.01, 0.05, 0.1, 0.5, 1, 5, 10], cv=5, max_iter=10000)
lasso_cv.fit(X_train, y_train)

print("\nLassoCV Results:")
print(f"Best alpha (LassoCV): {lasso_cv.alpha_}")
print(f"MSE (LassoCV): {mean_squared_error(y_test, lasso_cv.predict(X_test)):.4f}")
print(f"Non-zero coefficients: {np.sum(lasso_cv.coef_ != 0)}")

## Coefficient Path Visualization

In [ ]:
# Create coefficient path for different alphas
alphas = [0.001, 0.01, 0.05, 0.1, 0.5, 1, 5, 10]
coef_path = []

for alpha in alphas:
    lasso_temp = Lasso(alpha=alpha, max_iter=10000)
    lasso_temp.fit(X_train, y_train)
    coef_path.append(lasso_temp.coef_)

coef_path = np.array(coef_path)

plt.figure(figsize=(10, 6))
for i in range(n_features):
    plt.plot(alphas, coef_path[:, i], marker='o', label=f'Feature {i+1}')

plt.xscale('log')
plt.xlabel('Alpha (log scale)')
plt.ylabel('Coefficient Value')
plt.title('Lasso Coefficient Path vs Alpha')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Effect of Alpha on Feature Selection

In [ ]:
print("\n" + "="*50)
print("Effect of Alpha on Feature Selection")
print("="*50)

for alpha in alphas:
    lasso_temp = Lasso(alpha=alpha, max_iter=10000)
    lasso_temp.fit(X_train, y_train)
    non_zero = np.sum(np.abs(lasso_temp.coef_) > 1e-4)
    mse = mean_squared_error(y_test, lasso_temp.predict(X_test))
    print(f"Alpha = {alpha:5}: Non-zero = {non_zero:2}/{n_features}, MSE = {mse:.4f}")

## Cross-Validation Scores

In [ ]:
cv_scores = cross_val_score(best_lasso, X_scaled, y, cv=5, scoring='r2')

print("\n" + "="*50)
print("Cross-Validation Results")
print("="*50)
print(f"CV Scores: {cv_scores}")
print(f"Mean CV Score: {cv_scores.mean():.4f}")
print(f"Std CV Score: {cv_scores.std():.4f}")

In [ ]:
# Day Completed
print("\n" + "="*50)
print("LASSO REGRESSION - SCIKIT-LEARN COMPLETED")
print("="*50)
print("Topics covered:")
print("- Lasso Regression with Scikit-Learn")
print("- Linear Regression baseline comparison")
print("- Lasso vs Ridge comparison")
print("- GridSearchCV for alpha selection")
print("- LassoCV for built-in cross-validation")
print("- Coefficient path visualization")
print("- Effect of alpha on feature selection")
print("- Cross-validation scores")
print("="*50)"
print("\n*** LASSO REGRESSION MODULE COMPLETED ***")